# NB_01_READING_POINT

This notebook converts the reviewed output of `NB_00_SOURCE_EXTRACTION.ipynb` into cumulative RP_37 YAML specifications.

```text
NB_00_SOURCE_EXTRACTION
        ↓
reviewed source record
        ↓
NB_01_READING_POINT
        ↓
RP_37_A.yaml
RP_37_B.yaml
RP_37_C.yaml
        ↓
NB_TEMPLATE
```

The visible dialogue labels are source-specific. The Reading Point grammar remains in metadata.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import zipfile

try:
    import yaml
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "pyyaml"],
        check=True,
    )
    import yaml

NOTEBOOK_ID = "NB_01_READING_POINT"
NOTEBOOK_VERSION = "1.0.0"

SOURCE_DIRECTORY = Path("outputs/source_extraction/becker_2025_arpa_e")
SOURCE_RECORD = SOURCE_DIRECTORY / "becker_2025_source_extraction.yaml"

OUTPUT_DIRECTORY = Path("outputs/reading_points/RP_37")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

{
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "source_record": str(SOURCE_RECORD),
    "output_directory": str(OUTPUT_DIRECTORY),
}


## Load Reviewed Source Record

Run `NB_00_SOURCE_EXTRACTION.ipynb` first. This notebook consumes its YAML output.


In [ ]:
if not SOURCE_RECORD.exists():
    raise FileNotFoundError(
        f"Run NB_00_SOURCE_EXTRACTION first. Missing: {SOURCE_RECORD}"
    )

source_record = yaml.safe_load(SOURCE_RECORD.read_text(encoding="utf-8"))

if not isinstance(source_record, dict):
    raise TypeError("Source record must contain one top-level mapping")

source = source_record["source"]
candidate = source_record["reading_point_candidate"]

{
    "source_id": source["source_id"],
    "reading_point_id": candidate["reading_point_id"],
    "dialogue_count": len(candidate["dialogue"]),
}


## Review Candidate Dialogue

These are the engineering labels that will appear in the figures.


In [ ]:
for item in candidate["dialogue"]:
    print(item["order"], item["title"])
    print(" ", item["first_label"])
    print("   ↓")
    print(" ", item["second_label"])
    print("  support:", " | ".join(item["supporting_context"]))
    print()


## Canonical Repository Grammar

In [ ]:
REPOSITORY_GRAMMAR = [
    "Engineering Object specifies Engineering System.",
    "Engineering System produces Measured Engineering States.",
    "Measurement records Measured Engineering States.",
    "Measured Engineering States identify Engineering Constraints.",
    "Engineering Constraints direct Engineering Refinements.",
    "Engineering Refinements support Measured Engineering Improvement.",
    "Measured Engineering Improvement informs Leading Specifications.",
    "Leading Specifications direct Engineering Priorities.",
    "Engineering Priorities prepare Engineering Sessions.",
    "Engineering Sessions produce Engineering Records.",
    "Engineering Records support Engineering Reports.",
    "Engineering Reports support Repository Contributions.",
    "Repository Contributions support Repository Development.",
    "Repository Development supports Continued Specification.",
    "Continued Specification supports Engineering Object.",
]

GRAMMAR_BY_STAGE = {
    "A": "Measured Engineering Improvement informs Leading Specifications.",
    "B": "Leading Specifications direct Engineering Priorities.",
    "C": "Engineering Priorities prepare Engineering Sessions.",
}


## Build Cumulative RP_37 Specifications

A contains dialogue A. B contains A–B. C contains A–C.


In [ ]:
def required(mapping: dict[str, Any], key: str, context: str) -> Any:
    value = mapping.get(key)
    if value in (None, "", (), []):
        raise ValueError(f"{context}.{key} is required")
    return value


def normalized_dialogue(item: dict[str, Any], status: str) -> dict[str, Any]:
    order = str(required(item, "order", "dialogue"))
    support = list(required(item, "supporting_context", f"dialogue {order}"))

    if len(support) != 2:
        raise ValueError(
            f"dialogue {order}.supporting_context must contain two labels"
        )

    return {
        "order": order,
        "artifact_id": str(required(item, "artifact_id", f"dialogue {order}")),
        "concept": str(required(item, "concept", f"dialogue {order}")),
        "title": str(required(item, "title", f"dialogue {order}")),
        "first_label": str(required(item, "first_label", f"dialogue {order}")),
        "second_label": str(required(item, "second_label", f"dialogue {order}")),
        "supporting_context": [str(label) for label in support],
        "engineering_statement": str(
            required(item, "engineering_statement", f"dialogue {order}")
        ),
        "status": status,
    }


stage_configuration = {
    "A": {
        "notebook_id": "NB_37_A_DETECTOR_PERFORMANCE_TARGETS",
        "inherited_from": "NB_29_C_MEASURED_ENGINEERING_IMPROVEMENT",
        "natural_foundation": "Measured Engineering Improvement",
        "engineering_objective": "Specify source-derived detector-performance targets.",
        "forward_context": "Engineering Priorities",
        "completion_status": "developing",
    },
    "B": {
        "notebook_id": "NB_37_B_DETECTOR_REFINEMENT_PRIORITIES",
        "inherited_from": "NB_37_A_DETECTOR_PERFORMANCE_TARGETS",
        "natural_foundation": "Leading Specifications",
        "engineering_objective": "Direct detector-refinement priorities from source-derived targets.",
        "forward_context": "Engineering Sessions",
        "completion_status": "developing",
    },
    "C": {
        "notebook_id": "NB_37_C_ENGINEERING_SESSIONS",
        "inherited_from": "NB_37_B_DETECTOR_REFINEMENT_PRIORITIES",
        "natural_foundation": "Engineering Priorities",
        "engineering_objective": "Prepare fabrication, characterization, and validation sessions.",
        "forward_context": "Engineering Records",
        "completion_status": "complete",
    },
}

dialogue_source = candidate["dialogue"]
if [item["order"] for item in dialogue_source] != ["A", "B", "C"]:
    raise ValueError("RP_37 candidate dialogue must be ordered A, B, C")

specifications = {}

for stage_index, stage in enumerate(("A", "B", "C"), start=1):
    config = stage_configuration[stage]
    included = []

    for index, item in enumerate(dialogue_source[:stage_index]):
        item_status = "candidate" if index == stage_index - 1 else "admitted"
        included.append(normalized_dialogue(item, item_status))

    specifications[stage] = {
        "identity": {
            "notebook_id": config["notebook_id"],
            "reading_point": "RP_37",
            "stage": stage,
            "version": "1.0.0",
            "status": "candidate",
        },
        "reading_point": {
            "inherited_from": config["inherited_from"],
            "natural_foundation": config["natural_foundation"],
            "engineering_objective": config["engineering_objective"],
            "engineering_statements": [
                GRAMMAR_BY_STAGE[item["order"]] for item in included
            ],
            "repository_grammar": REPOSITORY_GRAMMAR,
            "forward_context": config["forward_context"],
            "status": config["completion_status"],
        },
        "dialogue": included,
        "engineering_object": source["engineering_object"],
        "engineering_direction": source["engineering_direction"],
        "source": {
            "source_id": source["source_id"],
            "title": source["title"],
            "author": source["author"],
            "organization": source["organization"],
            "event": source["event"],
            "date": source["date"],
            "source_file": source["source_file"],
        },
        "footer": "Admissible generalizations trail leading specifications.",
    }

specifications


## Export RP_37 A/B/C YAML

In [ ]:
generated_paths = []

for stage in ("A", "B", "C"):
    path = OUTPUT_DIRECTORY / f"RP_37_{stage}.yaml"
    path.write_text(
        yaml.safe_dump(
            specifications[stage],
            sort_keys=False,
            allow_unicode=True,
            width=100,
        ),
        encoding="utf-8",
    )
    generated_paths.append(path)

readme_path = OUTPUT_DIRECTORY / "README.md"
readme_path.write_text(
    "# RP_37 — Source-Derived Microcalorimeter Reading Point\n\n"
    "A: current detector performance → target detector performance\n\n"
    "B: target detector performance → detector-refinement priorities\n\n"
    "C: detector-refinement priorities → engineering sessions\n",
    encoding="utf-8",
)
generated_paths.append(readme_path)

zip_path = OUTPUT_DIRECTORY / "RP_37_SOURCE_DERIVED_YAML.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in generated_paths:
        archive.write(path, arcname=path.name)

generated_paths.append(zip_path)
generated_paths


## Preview Generated Dialogues

In [ ]:
for stage in ("A", "B", "C"):
    current = specifications[stage]["dialogue"][-1]

    print(f"RP_37_{stage}")
    print(current["title"])
    print(current["first_label"])
    print("↓")
    print(current["second_label"])
    print("support:", " | ".join(current["supporting_context"]))
    print()


## Verification

In [ ]:
expected_counts = {"A": 1, "B": 2, "C": 3}

for stage in ("A", "B", "C"):
    path = OUTPUT_DIRECTORY / f"RP_37_{stage}.yaml"

    if not path.exists():
        raise FileNotFoundError(path)
    if path.stat().st_size <= 0:
        raise ValueError(f"Empty output: {path}")

    loaded = yaml.safe_load(path.read_text(encoding="utf-8"))
    actual_count = len(loaded["dialogue"])

    if actual_count != expected_counts[stage]:
        raise ValueError(
            f"RP_37_{stage} dialogue count {actual_count}; "
            f"expected {expected_counts[stage]}"
        )

if not zip_path.exists() or zip_path.stat().st_size <= 0:
    raise ValueError("Reading Point ZIP was not created")

print("RP_37 source-derived YAML bundle: VERIFIED")
for path in generated_paths:
    print(f"{path} ({path.stat().st_size} bytes)")
